# 🎓 수학교육평가론 — 협력학습 MVP (v5)

**Human 1명 + AI 학생 3명(민준·서연·연우)** 이 '소수와 합성수'를 함께 탐구하는 협력학습 세션.
학습자모델(v2.0) · 교수자모델을 구현합니다.

```
User ──▶ 학습자 분석(01) ──▶ Learner Model(v2.0) ──▶ Tutor Decision(02) ──▶ AI 학생 발화(03)
```

## v5.1 변화 — 토큰 비용 감축
- **기본 LLM 공급자를 Google Gemini 2.0 Flash로 전환** — 무료 티어 활용 (15 RPM, 1500 RPD)
- `bootstrap(provider="gemini"|"anthropic")` 로 공급자 선택. Claude도 그대로 사용 가능.
- API 키: Google AI Studio에서 발급 (https://aistudio.google.com/app/apikey)

## v5 핵심 변화
- **모델 파라미터화** — `bootstrap(model=...)` 로 공급자별 모델을 자유롭게 지정

## v4 핵심 변화
- **학습자모델 v2.0**: 인지 2개(task_achievement·math_communication) + 정의 2개(cps·self_efficacy) + 확장 슬롯
- **3개 과제 구조**: Stage 1 개념 설명 → Stage 2 소수 찾기(30~50, 함정 33/39/49) → Stage 3 적응형 교과서 예제
- **Gradio UI 자동 대시보드 갱신**: 턴마다 오른쪽 6개 탭(레이더·학습자모델·변화추이·교수자 디시젼·오개념 타임라인·CPS 히트맵)이 자동 재렌더


## 1️⃣ 의존성 설치 + 저장소 준비

- `google-genai`, `gradio` 설치 (기본 공급자 = Gemini)
- (선택) Claude도 병행 사용하려면 `anthropic` 도 설치
- GitHub에서 코드 clone (최초 1회) 또는 `git pull` (재실행)
- **한글 폰트 NanumGothic** 은 `fonts/` 폴더에 동봉되어 별도 설치 불필요


In [ ]:
# 셀 1: 라이브러리 + 코드 저장소
\!pip install google-genai gradio -q
# (선택) Claude를 쓰고 싶을 때만:
# \!pip install anthropic -q

import os
REPO_URL = "https://github.com/hi-math/tobagi_v1.0.git"
REPO_DIR = "tobagi_v1_0"
if os.path.isdir(REPO_DIR):
    \!cd {REPO_DIR} && git pull -q
    print(f"🔄 기존 저장소 업데이트: {REPO_DIR}")
else:
    \!git clone -q {REPO_URL} {REPO_DIR}
    print(f"⬇️  저장소 클론 완료: {REPO_DIR}")


## 2️⃣ 경로 · 공급자 · 모델 설정

로컬 환경에서는 `BASE_PATH`를 저장소 절대경로로 바꾸세요 (예: `/content/drive/MyDrive/평가론 과제`).

**공급자/모델 선택**
- `PROVIDER = "gemini"` (기본, 무료 티어): `MODEL = "gemini-2.5-flash"` 또는 `"gemini-2.5-flash-lite"`(더 빠름), `"gemini-2.5-flash"`(품질)
- `PROVIDER = "anthropic"`: `MODEL = "claude-haiku-4-5-20251001"` (빠름) / `"claude-sonnet-4-20250514"` (품질)


In [ ]:
# 셀 2: 경로 · 공급자 · 모델
# -----------------------------------------------------------------
# PROVIDER:
#   - "gemini"    → Google Gemini (무료 티어, 기본 권장)
#   - "anthropic" → Anthropic Claude
#
# MODEL (비워두면 공급자별 기본값 사용):
#   Gemini:
#     - "gemini-2.5-flash"       (기본, 무료 티어 포함)
#     - "gemini-2.5-flash-lite"  (더 빠름/저렴)
#     - "gemini-2.5-flash"       (품질 우선, 유료)
#   Anthropic:
#     - "claude-haiku-4-5-20251001"
#     - "claude-sonnet-4-20250514"
# -----------------------------------------------------------------
BASE_PATH = "tobagi_v1_0"               # 저장소 루트 (상대/절대 모두 가능)
PROVIDER  = "gemini"                    # 기본: Gemini (무료 티어)
MODEL     = "gemini-2.5-flash"          # Gemini 기본
# PROVIDER = "anthropic";  MODEL = "claude-haiku-4-5-20251001"  # Claude를 쓰려면

import sys, os
_abs = os.path.abspath(BASE_PATH)
if _abs not in sys.path:
    sys.path.insert(0, _abs)

print("BASE_PATH=" + repr(BASE_PATH) + "  (abs: " + _abs + ")")
print("PROVIDER=" + repr(PROVIDER) + "  MODEL=" + repr(MODEL))


## 3️⃣ 부트스트랩 — 설정 로드 + 학습자모델 초기화 + LLM API 준비

`bootstrap()` 한 번으로 `CONFIG`(학습자·교수자·과제·도메인) + `PROMPTS`(프롬프트 템플릿) + `learner_models`(user + ai_1~3) + `api`(Gemini/Claude 래퍼) 네 가지를 받습니다.

**API 키 주입**: Colab Secrets에 `GEMINI_API_KEY`(또는 Claude 쓰는 경우 `CLAUDE_API_KEY`)를 저장 후 `userdata.get(...)`로 꺼내는 것을 권장합니다. 노트북에 직접 적어도 되지만 공유 시 주의.

Gemini 키 발급: https://aistudio.google.com/app/apikey


In [ ]:
# 셀 3: 부트스트랩
from google.colab import userdata
from lib import bootstrap

# 키는 Colab Secrets 사용을 권장. 직접 문자열로 넣어도 됨.
if PROVIDER == "gemini":
    apikey = userdata.get("GEMINI_API_KEY")  # 없으면 '' 반환 가능 — 그땐 문자열로 직접 입력
else:
    apikey = userdata.get("CLAUDE_API_KEY")

if not apikey:
    apikey = ""   # ← 여기에 API 키 문자열을 직접 붙여넣어도 됩니다

ctx = bootstrap(
    base_path=BASE_PATH,
    api_key=apikey,
    provider=PROVIDER,        # "gemini" (기본) 또는 "anthropic"
    model=MODEL,              # 셀 2에서 지정한 모델
    setup_fonts=True,         # NanumGothic 폰트 matplotlib 등록
)

CONFIG         = ctx["config"]
PROMPTS        = ctx["prompts"]
learner_models = ctx["learner_models"]
api            = ctx["api"]

print("Task: " + CONFIG["tasks"]["task_title"])
print("Stages: " + str(len(CONFIG["tasks"]["stages"])))
print("AI students: " + str([p["name"] for p in CONFIG["personas"]["ai_students"].values()]))
print("Learner model schema v" + str(CONFIG["learner_model_schema"].get("schema_version","?")))
print("  categories: " + str(list(CONFIG["learner_model_schema"].get("categories", {}).keys())))
print("  models: " + str(list(CONFIG["learner_model_schema"]["models"].keys())))
print("Prompts: " + str(list(PROMPTS.keys())))
print("Learner instances: " + str(list(learner_models.keys())))
print("Provider: " + str(api.provider) + "  Model: " + str(api.model))


## 4️⃣ 학습자 모델 v2.0 구조 확인

초기화된 사용자 학습자 모델을 Markdown으로 렌더해 v2.0 스키마가 정상 반영되었는지 확인합니다.


In [ ]:
# 셀 4: 학습자 모델 스냅샷
from IPython.display import Markdown, display
from lib import user_model_markdown

# 사용자 모델 초기 상태
display(Markdown(user_model_markdown(CONFIG, learner_models)))

# AI 학생 페르소나도 한 눈에
for aid, info in CONFIG["personas"]["ai_students"].items():
    print(f"🤖 {aid} — {info['name']} ({info['level']}/{info['role']})")
    print(f"   · {info['description']}")


## 5️⃣ Gradio 대화형 UI 실행

- **왼쪽**: 채팅(나 ↔ 민준·서연·연우) + `▶ 다음 Stage` 버튼
- **오른쪽 탭**: ① 🕸️ 레이더 · ② 🧠 학습자 모델 · ③ 📈 변화 추이 · ④ 🧭 교수자 디시젼 · ⑤ ⏱️ 오개념 타임라인 · ⑥ 🤝 CPS 히트맵
- **자동 갱신**: 한 턴이 끝날 때마다 6개 탭이 전부 재렌더됩니다.
- **침묵 유도**: 사용자가 60초 이상 말하지 않으면 AI 한 명이 먼저 말을 겁니다.
- Colab에서는 `share=True`로 공용 링크(~72시간) 생성.


In [ ]:
# 셀 5: Gradio UI 기동
from lib import launch_ui

launch_ui(**ctx, share=True)


## 6️⃣ (선택) 학습자 모델 종합 시각화 — 노트북 렌더

세션이 끝난 후 최종 학습자 모델을 Jupyter/Colab 셀에서 직접 렌더하고 싶을 때 사용합니다.

- `user_model_markdown` — 카테고리별 Stage 등급·루브릭 힌트·CPS 파생 레벨
- `plot_radar_all` — ordinal 차원 레이더
- `plot_user_history` — 회차별 변화 추이
- `cps_heatmap_figure` — Stage × CPS 4하위구인 누적 히트맵
- `misconception_timeline_figure` — 오개념 등장·소거 간트


In [ ]:
# 셀 6: 종합 시각화
from IPython.display import Markdown, display
from lib.visualize import (
    user_model_markdown, plot_radar_all, plot_user_history,
    cps_heatmap_figure, misconception_timeline_figure,
)
import matplotlib.pyplot as plt

# (a) Markdown 요약
display(Markdown(user_model_markdown(CONFIG, learner_models)))

# (b) 레이더
plot_radar_all(CONFIG, learner_models)

# (c) 변화 추이
plot_user_history(CONFIG, learner_models)

# (d) CPS 히트맵
fig_cps = cps_heatmap_figure(CONFIG, learner_models)
plt.show()

# (e) 오개념 타임라인
fig_mis = misconception_timeline_figure(CONFIG, learner_models)
plt.show()


---

## 🗂️ (선택) 레거시 CLI 런너

Gradio UI 대신 터미널 상호작용으로 세션을 돌리고 싶을 때.

명령어: `/radar` · `/history` · `/model` · `/next` · `/decision` · `/quit`


In [ ]:
# 셀 7 (선택): CLI 세션
from lib import run_session

# run_session(**ctx)   # 주석 해제 후 실행
